In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
# Delete the past output
import shutil
from pathlib import Path

out = Path("outputs/smol-lora")
if out.exists():
  shutil.rmtree(out)

In [3]:
import os, shutil
from kaggle_secrets import UserSecretsClient

# Secrets
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# Pull latest code
repo_path = "/kaggle/working/mini-artificer"
if os.path.exists(repo_path):
    %cd {repo_path}
    !git pull origin main
else:
    !git clone https://github.com/reuel-ly/mini-artificer.git {repo_path}
    %cd {repo_path}

# Install dependencies
!pip install -q transformers peft trl datasets bitsandbytes accelerate wandb

print("Ready.")

Cloning into '/kaggle/working/mini-artificer'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 206 (delta 2), reused 6 (delta 2), pack-reused 194 (from 1)
Receiving objects: 100% (206/206), 111.67 KiB | 1.51 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/kaggle/working/mini-artificer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 110.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is 

In [4]:
!python train.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mpornobereuel (mpornobereuel-reuel) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/mini-artificer/wandb/run-20260625_142425-h2af9gu3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run smollm2-135m-lora-r4-10k-curated-7000steps-2
wandb: ⭐️ View project at https://wandb.ai/mpornobereuel-reuel/mini-artificer
wandb: 🚀 View run at https://wandb.ai/mpornobereuel-reuel/mini-artificer/runs/h2af9gu3
README.md: 100%|████████████████████████████████| 106/106 [00:00<00:00, 381kB/s]
glaive-function-calling-v2.json: 100%|████████| 271M/271M [00:02<00:00, 121MB/s]
config.json: 100%|█████████████████████████████| 861/861 [00:00<

In [5]:
import os
print(os.path.exists("/kaggle/working/mini-artificer/outputs/smol-lora"))

True


In [6]:
 # 1. List output dir
import os
print(os.listdir("/kaggle/working/mini-artificer/outputs/smol-lora"))
# 2. Check chat template was saved
import json
with open("/kaggle/working/mini-artificer/outputs/smol-lora/tokenizer_config.json") as f:
  cfg = json.load(f)
print("Has generation markers:", "{% generation %}" in cfg.get("chat_template", ""))
# 3. Raw input debug (add to inference.py or run inline)
from inference import _load_model_and_tokenizer, WEATHER_TOOL_SCHEMA
from transformers import AutoTokenizer
output_dir = "/kaggle/working/mini-artificer/outputs/smol-lora"
_, tokenizer = _load_model_and_tokenizer(output_dir)
messages = [
  {"role": "system", "content": "You are a helpful assistant with access to the following functions. Use them if required -\n" + str(WEATHER_TOOL_SCHEMA)},
  {"role": "user", "content": "What is the weather in Manila?"},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt")
print("=== RAW INPUT ===")
print(tokenizer.decode(inputs["input_ids"][0]))
print("=================")

['checkpoint-500', 'checkpoint-300', 'checkpoint-1200', 'checkpoint-1000', 'checkpoint-1300', 'checkpoint-1600', 'checkpoint-200', 'checkpoint-1400', 'checkpoint-1100', 'checkpoint-1500', 'adapter_model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'checkpoint-1700', 'README.md', 'chat_template.jinja', 'checkpoint-800', 'checkpoint-100', 'checkpoint-700', 'checkpoint-900', 'checkpoint-400', 'checkpoint-600', 'adapter_config.json', 'tokenizer.json']
Has generation markers: False


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

=== RAW INPUT ===
<|im_start|>system
You are a helpful assistant with access to the following functions. Use them if required -
{'name': 'get_weather', 'description': 'Get current weather for a location', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'City name'}}, 'required': ['location']}}<|im_end|>
<|im_start|>user
What is the weather in Manila?<|im_end|>
<|im_start|>assistant



In [7]:
# Check what's actually saved in your output directory
import os
print(os.listdir("/kaggle/working/mini-artificer/outputs/smol-lora"))

# Load and print the saved chat template
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(
    "/kaggle/working/mini-artificer/outputs/smol-lora"
)
print(tok.chat_template)

['checkpoint-500', 'checkpoint-300', 'checkpoint-1200', 'checkpoint-1000', 'checkpoint-1300', 'checkpoint-1600', 'checkpoint-200', 'checkpoint-1400', 'checkpoint-1100', 'checkpoint-1500', 'adapter_model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'checkpoint-1700', 'README.md', 'chat_template.jinja', 'checkpoint-800', 'checkpoint-100', 'checkpoint-700', 'checkpoint-900', 'checkpoint-400', 'checkpoint-600', 'adapter_config.json', 'tokenizer.json']
{% for message in messages %}{% if message['role'] == 'system' %}<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}<|im_start|>assistant
{% generation %}{{ message['content'] }}{% endgeneration %}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant
{% endif %}


In [8]:
from transformers import AutoTokenizer
from chat_template import patch_chat_template

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
tokenizer = patch_chat_template(tokenizer)

sample_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "What is the weather in Manila?"},
    {"role": "assistant", "content": '<functioncall> {"name": "get_weather", "arguments": {"location": "Manila"}}'}
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
)
print(formatted)

<|im_start|>system
You are helpful.<|im_end|>
<|im_start|>user
What is the weather in Manila?<|im_end|>
<|im_start|>assistant
<functioncall> {"name": "get_weather", "arguments": {"location": "Manila"}}<|im_end|>



In [9]:
print("eos:", tokenizer.eos_token_id, tokenizer.eos_token)
print("im_end:", tokenizer.convert_tokens_to_ids(""))
print("eot:", tokenizer.convert_tokens_to_ids("<|endoftext|>"))

eos: 2 <|im_end|>
im_end: 0
eot: 0
